<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day08-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 8, Segment 1 discussion — Does more folds always help cross-validation?

The book page measured 5-fold cross-validation on the subcellular-localization
dataset and found its fold-to-fold standard deviation (0.039) was *not* smaller
than the standard deviation across ten single random splits (0.031) — a
genuinely small-dataset effect, not a mistake.

**This notebook asks the natural follow-up: does using more folds fix that?**
We refetch the same live UniProt dataset and compare k=3, k=5, and k=10-fold
cross-validation on it directly.

In [1]:
import numpy as np
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split

np.random.seed(0)

AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
SEQ_LEN = 150

def one_hot_encode(sequence, length=SEQ_LEN):
    encoded = np.zeros((length, len(AMINO_ACIDS)), dtype=np.float32)
    for position, residue in enumerate(sequence[:length]):
        if residue in AA_TO_INDEX:
            encoded[position, AA_TO_INDEX[residue]] = 1.0
    return encoded.flatten()

UNIPROT_CLASSES = {
    "Cytoplasm": "SL-0086",
    "Nucleus": "SL-0191",
    "Mitochondrion": "SL-0173",
    "Secreted": "SL-0243",
    "Cell membrane": "SL-0039",
}
CLASS_NAMES = list(UNIPROT_CLASSES.keys())

def fetch_uniprot(sl_code, size=500):
    query = f"organism_id:9606 AND reviewed:true AND cc_scl_term:{sl_code} AND length:[50 TO 500]"
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/search",
        params={"query": query, "fields": "accession,sequence,cc_subcellular_location",
                "format": "tsv", "size": size},
        timeout=60,
    )
    r.raise_for_status()
    lines = r.text.strip().split("\n")[1:]
    seqs = []
    for line in lines:
        parts = line.split("\t")
        if len(parts) >= 2 and parts[1]:
            seqs.append(parts[1])
    return seqs

print("Fetching the same 5 classes live from UniProt (this takes a moment)...")
sequences_by_class = {name: fetch_uniprot(code) for name, code in UNIPROT_CLASSES.items()}
for name, seqs in sequences_by_class.items():
    print(f"  {name}: {len(seqs)} sequences")

Fetching the same 5 classes live from UniProt (this takes a moment)...


  Cytoplasm: 500 sequences
  Nucleus: 500 sequences
  Mitochondrion: 500 sequences
  Secreted: 500 sequences
  Cell membrane: 500 sequences


In [2]:
N_PER_CLASS = min(len(v) for v in sequences_by_class.values())
print("balancing every class to", N_PER_CLASS, "sequences")

rng = np.random.RandomState(0)
balanced_seqs, balanced_labels = [], []
for class_idx, class_name in enumerate(CLASS_NAMES):
    pool = sequences_by_class[class_name]
    chosen = rng.choice(len(pool), N_PER_CLASS, replace=False)
    for i in chosen:
        balanced_seqs.append(pool[i])
        balanced_labels.append(class_idx)

X = np.stack([one_hot_encode(s) for s in balanced_seqs])
y = np.array(balanced_labels)
print("total balanced dataset:", len(X), "sequences,", len(CLASS_NAMES), "classes")

# Hold out a test set exactly as the book page does, and cross-validate only
# on the remaining train+val pool.
X_cv, X_test, y_cv, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=0)
print("cross-validation pool:", len(X_cv), " held-out test (untouched):", len(X_test))

balancing every class to 500 sequences
total balanced dataset: 2500 sequences, 5 classes
cross-validation pool: 2125  held-out test (untouched): 375


## k=3, k=5, k=10 fold cross-validation, same data, same model

Each run partitions the *same* `X_cv`/`y_cv` pool differently — more folds
means each fold is smaller (less held out per fold) but there are more of
them.

In [3]:
results = {}
for k in [3, 5, 10]:
    kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)
    fold_accuracies = []
    for train_idx, val_idx in kfold.split(X_cv, y_cv):
        clf = LogisticRegression(max_iter=2000, random_state=0)
        clf.fit(X_cv[train_idx], y_cv[train_idx])
        fold_accuracies.append(clf.score(X_cv[val_idx], y_cv[val_idx]))
    fold_accuracies = np.array(fold_accuracies)
    results[k] = fold_accuracies
    print(f"k={k:2d}: fold accuracies = {np.round(fold_accuracies, 3)}")
    print(f"       mean = {fold_accuracies.mean():.3f}   std = {fold_accuracies.std():.3f}   "
          f"(avg. fold size = {len(X_cv)//k})")

k= 3: fold accuracies = [0.515 0.49  0.492]
       mean = 0.499   std = 0.011   (avg. fold size = 708)


k= 5: fold accuracies = [0.544 0.482 0.515 0.511 0.489]
       mean = 0.508   std = 0.022   (avg. fold size = 425)


k=10: fold accuracies = [0.512 0.507 0.479 0.521 0.596 0.491 0.524 0.514 0.472 0.557]
       mean = 0.517   std = 0.035   (avg. fold size = 212)


## What actually happened, read honestly

Look at the printed means and stds above rather than assuming "more folds =
smaller std". More folds means each validation fold is smaller (for k=10,
each fold is roughly `len(X_cv)/10` proteins), which makes each individual
fold's accuracy *noisier*, not less — even though there are more of them to
average over. Whether the overall std goes down, stays flat, or even goes up
between k=5 and k=10 on this specific ~600-protein dataset is an empirical
question this cell just answered for real, not a guaranteed textbook trend.
The one thing that *does* reliably improve with larger k: every example gets
validated on more often relative to dataset size, and the *mean* accuracy
estimate (not necessarily its std) tends to stabilize.

**Discuss:** given what you just measured, would you recommend k=10 over k=5
for a dataset this size? What would you need to change about the dataset (or
the experiment) to get a cleaner answer?